In [1]:
import sys
import os

# Ruta ubicacion scripts mhw
ruta_mhw = os.path.abspath(os.path.join('..', 'mhw'))

# Agregamos la ruta al sistema
if ruta_mhw not in sys.path:
    sys.path.append(ruta_mhw)

import warnings
# Ignora warnings especificamente de numpy y la existencia de arrays con nans
#warnings.filterwarnings("ignore", message="All-NaN slice encountered")

%load_ext autoreload
%autoreload 2

In [13]:
import numpy as np
import xarray as xr
import pandas as pd

import datasets_utils as dsu
import mhw_core as mhw
import time_utils as timeu

nf_sst = "ncdcOisst21AggLat-55-335Lon-70-495_Till31dic24.nc"
nf_pcaso = "pcaso_clim.nc"
nf_waves = "pcaso_waves.nc"
nf_df_waves = "pcaso_df_waves.csv"

# Calculo de MHW

In [4]:
if dsu.exist_file(dsu.join_path(dsu.DIR_DATASETS, nf_pcaso)):
    ds_clim = xr.open_dataset(dsu.join_path(dsu.DIR_DATASETS, nf_pcaso))
else:
    ds = xr.open_dataset(dsu.join_path(dsu.DIR_SST, nf_sst))
    ds_clim = mhw.get_climatology_and_anomalies(ds.sst)
    dsu.save_dataset_as_nc(ds_clim, name_file=nf_pcaso)
    ds.close()

In [5]:
if dsu.exist_file(dsu.join_path(dsu.DIR_DATASETS, nf_waves)):
    waves = xr.open_dataset(dsu.join_path(dsu.DIR_DATASETS, nf_waves))
else:
    waves = mhw.get_marine_heat_waves(ds_clim.is_sup_p90)
    dsu.save_dataset_as_nc(waves, name_file=nf_waves)

In [20]:
if dsu.exist_file(dsu.join_path(dsu.DIR_DATASETS, nf_df_waves)):
    df_waves = pd.read_csv(dsu.join_path(dsu.DIR_DATASETS, nf_df_waves))
else:
    df_waves = mhw.waves_to_dataframe(waves.waves, anomalies=ds_clim.anomalias)
    dsu.save_dataframe_as_csv(df, name_file=nf_df_waves)